# ViMamba-SER — Phase F Training Notebook (Colab)

Notebook chạy trực tiếp trên Google Colab, bao gồm:
1. Mount Google Drive + clone code
2. Cài đặt dependencies (thử mamba-ssm, fallback nếu lỗi)
3. Tải dataset ViSEC từ HuggingFace
4. Trích sequence embeddings + cache vào Drive
5. Train Phase F (1 seed verify nhanh)
6. Chạy 5-fold CV cho A1, B3b, F
7. Lưu checkpoint + kết quả vào Drive

---

## Nếu Colab bị disconnect giữa chừng

| Bước đã xong | Resume từ cell |
|---|---|
| Chưa chạy gì | Cell 1 (mount Drive) |
| Đã mount Drive, clone code | Cell 3 (cài dependencies) |
| Đã cài dependencies | Cell 5 (tải dataset) |
| Đã trích embeddings (kiểm tra Drive có thư mục `sequence/`) | Cell 7 (train Phase F) |
| Đã train Phase F xong | Cell 8 (5-fold CV) |

Mọi embeddings và checkpoint đều cache trên Drive, nên không mất khi session ngắt.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ViMamba-SER-outputs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Output dir: {DRIVE_DIR}')

## 2. Clone/Pull code từ repo

Dùng **1 trong 2 cách** bên dưới (comment/uncomment tùy hoàn cảnh).

In [ ]:
# ============================================================
# CÁCH 1: Clone từ GitHub (ưu tiên dùng cách này)
# ============================================================
REPO_URL = 'https://github.com/QuangHuy1911/ViMamba-SER.git'
CODE_DIR = '/content/ViMamba-SER'

if os.path.exists(CODE_DIR):
    print('Repo đã tồn tại, pull latest...')
    !cd {CODE_DIR} && git pull
else:
    !git clone {REPO_URL} {CODE_DIR}

# ============================================================
# CÁCH 2: Upload zip thủ công (dùng nếu repo private / không có mạng)
# Uncomment 3 dòng dưới, upload file ViMamba-SER.zip khi được hỏi
# ============================================================
# from google.colab import files
# uploaded = files.upload()  # upload ViMamba-SER.zip
# !unzip -o ViMamba-SER.zip -d /content/ViMamba-SER

os.chdir(CODE_DIR)
print(f'Working directory: {os.getcwd()}')
!ls -la

## 3. Cài đặt dependencies cơ bản

In [ ]:
!pip install -q transformers datasets librosa soundfile tqdm pyyaml
!pip install -q scikit-learn matplotlib seaborn
print('Dependencies cơ bản đã cài xong.')

## 3b. Thử cài mamba-ssm (không bắt buộc)

Nếu cell này lỗi → hoàn toàn bình thường, code sẽ tự động dùng BiGRU fallback.

In [ ]:
MAMBA_INSTALLED = False
try:
    import mamba_ssm
    MAMBA_INSTALLED = True
    print(f'mamba-ssm đã có sẵn, version: {mamba_ssm.__version__}')
except ImportError:
    print('mamba-ssm chưa cài, thử cài...')
    try:
        import subprocess, sys
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install',
             'mamba-ssm[causal-conv1d]', '--no-build-isolation', '-q'],
            capture_output=True, text=True, timeout=600
        )
        if result.returncode == 0:
            import mamba_ssm
            MAMBA_INSTALLED = True
            print(f'Cài mamba-ssm thành công! Version: {mamba_ssm.__version__}')
        else:
            print(f'Cài mamba-ssm thất bại (exit code {result.returncode}).')
            print('Dùng fallback BiGRU fusion, bỏ qua mamba-ssm.')
            if result.stderr:
                # Chỉ in 10 dòng cuối để debug nếu cần
                print('--- Lỗi (10 dòng cuối) ---')
                print('\n'.join(result.stderr.strip().split('\n')[-10:]))
    except Exception as e:
        print(f'Lỗi khi cài mamba-ssm: {e}')
        print('Dùng fallback BiGRU fusion, bỏ qua mamba-ssm.')

print(f'\nTrạng thái: {"Bi-Mamba thật" if MAMBA_INSTALLED else "BiGRU fallback"}')

## 4. Verify GPU + import modules

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# Import project modules
import sys
sys.path.insert(0, '.')
from src.config import *
print(f'\nDevice: {DEVICE}')
print(f'Dataset: {DATASET_NAME}')
print(f'Num classes: {NUM_CLASSES}')

## 5. Tải dataset ViSEC từ HuggingFace

In [ ]:
from datasets import load_dataset

# Cache dataset vào Drive để không tải lại mỗi session
DATASET_CACHE = os.path.join(DRIVE_DIR, 'hf_cache')
os.makedirs(DATASET_CACHE, exist_ok=True)

print('Đang tải ViSEC dataset...')
ds = load_dataset(DATASET_NAME, cache_dir=DATASET_CACHE)
print(f'Số lượng samples: {len(ds["train"])}')
print(f'Features: {ds["train"].features}')
print(f'\nPhân bố nhãn:')
from collections import Counter
label_dist = Counter(ds['train']['label'])
for label, count in sorted(label_dist.items()):
    print(f'  {label}: {count}')

## 6. Trích xuất sequence-level embeddings + cache vào Drive

Bước này mất ~15-30 phút lần đầu. Nếu đã chạy trước đó, sẽ skip samples đã cache.

In [ ]:
import numpy as np
import json
from pathlib import Path
from tqdm.auto import tqdm

# Thư mục cache sequence embeddings trên Drive
SEQ_CACHE_DIR = os.path.join(DRIVE_DIR, 'embeddings', 'sequence')
os.makedirs(os.path.join(SEQ_CACHE_DIR, 'audio'), exist_ok=True)
os.makedirs(os.path.join(SEQ_CACHE_DIR, 'text'), exist_ok=True)

# Kiểm tra đã cache bao nhiêu
n_total = len(ds['train'])
cached_audio = len(list(Path(SEQ_CACHE_DIR, 'audio').glob('*.pt')))
cached_text = len(list(Path(SEQ_CACHE_DIR, 'text').glob('*.pt')))
print(f'Đã cache: {cached_audio}/{n_total} audio, {cached_text}/{n_total} text')

if cached_audio >= n_total and cached_text >= n_total:
    print('Tất cả embeddings đã có cache, skip extraction.')
else:
    print('Cần trích thêm embeddings...')

In [ ]:
# Cell này chỉ chạy nếu cần trích thêm
# Nếu đã có đủ cache ở cell trên, skip cell này

from transformers import WavLMModel, AutoProcessor, AutoModel, AutoTokenizer
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from src.features.extract_sequence_embeddings import (
    extract_wavlm_sequence, extract_phobert_sequence,
    cache_embedding, load_cached_embedding,
)
import torchaudio

device = DEVICE

# Load models (frozen)
print('Loading WavLM-Base...')
wavlm_processor = AutoProcessor.from_pretrained(WAVLM_CKPT)
wavlm_model = WavLMModel.from_pretrained(WAVLM_CKPT).to(device).eval()

print('Loading PhoWhisper-Base (ASR)...')
whisper_processor = WhisperProcessor.from_pretrained(PHOWHISPER_CKPT)
whisper_model = WhisperForConditionalGeneration.from_pretrained(
    PHOWHISPER_CKPT
).to(device).eval()

print('Loading PhoBERT-v2...')
phobert_tokenizer = AutoTokenizer.from_pretrained(PHOBERT_CKPT)
phobert_model = AutoModel.from_pretrained(PHOBERT_CKPT).to(device).eval()

print('Tất cả model đã load xong.')

# Lưu labels
labels_path = os.path.join(SEQ_CACHE_DIR, 'labels.npy')
transcripts_path = os.path.join(SEQ_CACHE_DIR, 'transcripts.json')
transcripts = {}

# Load transcripts đã có nếu resume
if os.path.exists(transcripts_path):
    with open(transcripts_path) as f:
        transcripts = json.load(f)

labels_all = []

for i, sample in enumerate(tqdm(ds['train'], desc='Extracting')):
    sample_id = f'sample_{i:05d}'
    
    # Label
    label_name = sample['label'] if isinstance(sample['label'], str) else LABEL_NAMES[sample['label']]
    labels_all.append(LABEL_MAP.get(label_name, sample['label']))
    
    # Skip nếu đã cache
    audio_cached = load_cached_embedding(sample_id, SEQ_CACHE_DIR, 'audio')
    text_cached = load_cached_embedding(sample_id, SEQ_CACHE_DIR, 'text')
    if audio_cached is not None and text_cached is not None:
        continue
    
    # Audio waveform
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    waveform = torch.tensor(audio_array, dtype=torch.float32)
    
    # Resample nếu cần
    if sr != SAMPLE_RATE:
        resampler = torchaudio.transforms.Resample(sr, SAMPLE_RATE)
        waveform = resampler(waveform)
    
    # WavLM sequence embedding
    if audio_cached is None:
        audio_emb = extract_wavlm_sequence(
            waveform, wavlm_model, wavlm_processor, device
        )
        cache_embedding(audio_emb, sample_id, SEQ_CACHE_DIR, 'audio')
    
    # Transcript bằng PhoWhisper
    if sample_id not in transcripts:
        whisper_inputs = whisper_processor(
            waveform.numpy(), sampling_rate=SAMPLE_RATE,
            return_tensors='pt'
        ).input_features.to(device)
        with torch.no_grad():
            generated_ids = whisper_model.generate(whisper_inputs)
            transcript = whisper_processor.batch_decode(
                generated_ids, skip_special_tokens=True
            )[0]
        transcripts[sample_id] = transcript
        # Lưu transcripts thường xuyên phòng disconnect
        if i % 100 == 0:
            with open(transcripts_path, 'w', encoding='utf-8') as f:
                json.dump(transcripts, f, ensure_ascii=False, indent=2)
    
    # PhoBERT sequence embedding
    if text_cached is None:
        text_emb = extract_phobert_sequence(
            transcripts[sample_id], phobert_model, phobert_tokenizer, device
        )
        cache_embedding(text_emb, sample_id, SEQ_CACHE_DIR, 'text')
    
    # Giải phóng VRAM định kỳ
    if i % 200 == 0:
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Lưu cuối cùng
np.save(labels_path, np.array(labels_all))
with open(transcripts_path, 'w', encoding='utf-8') as f:
    json.dump(transcripts, f, ensure_ascii=False, indent=2)

# Giải phóng model khỏi VRAM
del wavlm_model, whisper_model, phobert_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f'\nXong! Đã cache {n_total} samples vào {SEQ_CACHE_DIR}')
print(f'Labels: {labels_path}')
print(f'Transcripts: {transcripts_path}')

## 7. Train Phase F — 1 seed verify nhanh

Chạy 1 lần với seed=42 để verify pipeline hoạt động (~15-20 phút).

In [ ]:
# Đảm bảo đang ở đúng thư mục code
os.chdir(CODE_DIR)

# Chạy train Phase F
!python train.py \
    --phase f \
    --seed 42 \
    --epochs 50 \
    --batch_size 16 \
    --save_dir {DRIVE_DIR}/runs \
    --embed_dir {DRIVE_DIR}/embeddings

In [ ]:
# Nếu muốn so sánh nhanh với Phase A (audio-only baseline)
!python train.py \
    --phase a \
    --seed 42 \
    --epochs 50 \
    --batch_size 64 \
    --save_dir {DRIVE_DIR}/runs \
    --embed_dir {DRIVE_DIR}/embeddings

## 8. Chạy 5-fold CV cho A1, B3b, F

Bước này mất nhiều thời gian hơn (5 folds × 3 phases × 50 epochs).
Checkpoint mỗi fold được lưu riêng trên Drive.

In [ ]:
os.chdir(CODE_DIR)

!python run_5fold_cv.py \
    --phases a b f \
    --seed 42 \
    --epochs 50 \
    --batch_size 16 \
    --output {DRIVE_DIR}/results/5fold_cv_results.csv \
    --embed_dir {DRIVE_DIR}/embeddings \
    --save_dir {DRIVE_DIR}/runs

In [ ]:
# Xem kết quả
import pandas as pd

results_path = os.path.join(DRIVE_DIR, 'results', '5fold_cv_results.csv')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    print('=== 5-Fold CV Results ===')
    print(df.to_string(index=False))
else:
    print(f'File {results_path} chưa tồn tại. Kiểm tra lại cell trước.')

## 9. Lưu checkpoint + kết quả vào Drive

In [ ]:
import shutil

# Copy kết quả local vào Drive (nếu chưa lưu trực tiếp)
for src_dir, name in [('runs', 'runs'), ('reports', 'reports')]:
    local_dir = os.path.join(CODE_DIR, src_dir)
    drive_dest = os.path.join(DRIVE_DIR, name)
    if os.path.exists(local_dir):
        os.makedirs(drive_dest, exist_ok=True)
        for f in os.listdir(local_dir):
            src = os.path.join(local_dir, f)
            dst = os.path.join(drive_dest, f)
            if os.path.isfile(src):
                shutil.copy2(src, dst)
                print(f'Copied: {f}')

print(f'\nTất cả kết quả đã lưu vào: {DRIVE_DIR}')
print('Nội dung:')
!find {DRIVE_DIR} -type f | head -30

## 10. Chạy unit tests (verify code đúng trên Colab GPU)

In [ ]:
os.chdir(CODE_DIR)
!python -m unittest tests.test_tme_fusion -v